# Web Scraping Notebook

## Imports & Setup
Import necessary libraries and set up the environment for scraping.

In [ ]:
# Import libraries
import os
import time
import json
import random
import requests
import pandas as pd
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager

## Scraping Configuration
Define constants such as URLs, headers, input/output file paths, and scraping limits.

In [ ]:
HEADERS = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
}

## Scraping Functions

### Harvesting Real Estate listing links

In [ ]:
# base_url = "https://ly.opensooq.com/en/property/residential-for-sale?page="
base_url = "https://ly.opensooq.com/en/find?sort_code=recent&page="
page_number = 1
next_part = "&vertical_link=Property/Buy/Buy+Residential"
total_links_saved = 0

In [ ]:
def setup_driver():
    chrome_options = Options()
    chrome_options.add_argument("--headless")
    chrome_options.add_argument("--no-sandbox")
    chrome_options.add_argument("--disable-dev-shm-usage")
    driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=chrome_options)
    return driver

def harvest_links(base_url, next_part, max_pages, output_file):
    print("--- Starting Link Harvest ---")

    try:
        while (page_number <= max_pages):
            target_url = f"{base_url}{page_number}{next_part}"
            print(f"Scraping Page {page_number}...")
            
            driver.get(target_url)

            # 2. Wait for the cards to appear
            try:
                WebDriverWait(driver, 15).until(
                    EC.presence_of_all_elements_located((By.CLASS_NAME, "postListItemData"))
                )
            except Exception:
                # If we wait 15 seconds and no cards appear, we are either blocked or finished
                print(f"No listings found on page {page_number}. Checking if we are finished...")
                break

            # 3. Parse HTML
            soup = BeautifulSoup(driver.page_source, 'html.parser')
            cards = soup.select('a.postListItemData')

            if not cards:
                print("End of results reached.")
                break

            # 4. Save to file IMMEDIATELY (Append mode 'a')
            with open(output_file + ".txt", "a", encoding="utf-8") as file:
                for card in cards:
                    href = card.get('href')
                    if href:
                        full_link = "https://ly.opensooq.com" + href if not href.startswith('http') else href
                        file.write(full_link + "\n")
                        total_links_saved += 1
            
            print(f"Saved {len(cards)} links. Total so far: {total_links_saved}")

            # 5. Human-like behavior
            page_number += 1
            time.sleep(random.uniform(3, 6)) # Vital for avoiding IP bans

    except KeyboardInterrupt:
        print("\nProcess stopped by user. Progress saved.")
    except Exception as e:
        print(f"An error occurred: {e}")
    finally:
        driver.quit()
        print(f"--- Harvest Complete! Total links in file: {total_links_saved} ---")

### Scraping Harvested Links

In [ ]:
# Configuration
OPEN_SOUQ_INPUT_FILE = "all_listings_links.txt"
BAHU_INPUT_FILE = "bahu_links.csv"
OUTPUT_FILE = "scraped_data.csv"
START_INDEX = 1
END_INDEX = 6500

In [ ]:
def get_already_scraped():
    if not os.path.exists(OUTPUT_FILE):
        return set()
    try:
        with open(OUTPUT_FILE, 'r', encoding='utf-8') as f:
            data = json.load(f)
            return {item['url'] for item in data}
    except:
        return set()


def scrape_details(url):
    try:
        response = requests.get(url, headers=HEADERS, timeout=15)
        if response.status_code != 200: return None
        
        soup = BeautifulSoup(response.text, 'html.parser')
        
        # Initialize the dictionary
        property_data = {
            "url": url,
            "price": "N/A",
            "location": "N/A",
            "attributes": {}
        }

        # 1. EXTRACT PRICE
        # Looking for the div with class 'priceColor'
        price_tag = soup.find('div', class_='priceColor')
        if price_tag:
            property_data["price"] = price_tag.get_text(strip=True)

        # 2. EXTRACT GOOGLE MAPS LINK
        map_link_tag = soup.find('a', href=lambda x: x and ('google.com/maps' in x or 'googleusercontent.com' in x))

        if map_link_tag:
            property_data["location"] = map_link_tag['href']

        # 3. EXTRACT ATTRIBUTES
        info_section = soup.find('section', id='PostViewInformation')
        if info_section:
            items = info_section.find_all('li', {'data-id': lambda x: x and x.startswith('singeInfoField')})
            for item in items:
                key_tag = item.find('p')
                if key_tag:
                    key = key_tag.get_text(strip=True)
                    val_tag = item.find('a') or item.find('span')
                    property_data["attributes"][key] = val_tag.get_text(strip=True) if val_tag else "N/A"

        return property_data
    except Exception as e:
        print(f"Error scraping {url}: {e}")
        return None

## Execution Pipeline
Create a pipeline to execute the scraping process, including link harvesting, filtering, and detail extraction.

In [ ]:
def execute_pipeline():
    # Harvest OpenSouq links
    print("Harvesting OpenSouq links...")
    harvest_links(base_url, next_part, 10, OPEN_SOUQ_INPUT_FILE)

    # 1. Read all lines from file
    if not os.path.exists(INPUT_FILE):
        print(f"Error: {INPUT_FILE} not found!")
        return

    with open(INPUT_FILE, "r", encoding="utf-8") as f:
        all_links = [line.strip() for line in f if line.strip()]

    # 2. APPLY THE LIMIT/RANGE
    links_subset = all_links[START_INDEX:END_INDEX]
    
    # 3. Filter out links that are broken and links already in your JSON
    scraped_urls = get_already_scraped()
    links_to_process = [i for i in links_subset if i not in scraped_urls]
    
    print(f"File contains {len(all_links)} links.")
    print(f"Targeting range [{START_INDEX}:{END_INDEX}]. ({len(links_subset)} links).")
    print(f"After checking for duplicates, {len(links_to_process)} links left to scrape.")

    # 4. Load existing results
    results = []
    if os.path.exists(OUTPUT_FILE):
        with open(OUTPUT_FILE, 'r', encoding='utf-8') as f:
            try:
                results = json.load(f)
            except:
                results = []

    # 5. Loopie loopppppppp through the list
    for i, url in enumerate(links_to_process):
        print(f"Processing {i+1}/{len(links_to_process)}: {url}")
        
        data = scrape_details(url)
        if data:
            results.append(data)
            
            # Save every 5 dinars and items just in case
            if (i + 1) % 5 == 0:
                with open(OUTPUT_FILE, 'w', encoding='utf-8') as f:
                    json.dump(results, f, indent=4, ensure_ascii=False)

        time.sleep(random.uniform(2, 4))

    # Final Save!!!!!!
    with open(OUTPUT_FILE, 'w', encoding='utf-8') as f:
        json.dump(results, f, indent=4, ensure_ascii=False)
    print("Done!")

    # Save results
    df = pd.DataFrame(results)
    df.to_csv(OUTPUT_FILE, index=False)
    print("Pipeline execution complete.")

## Data Storage / Export
Save the scraped data to JSON or CSV files, ensuring data integrity and proper formatting.

In [ ]:
# Execute and save
execute_pipeline()